In [1]:
# 运行此代码单元格来挂载 Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os

# 成功后 Google Drive内容将可以在以下路径访问：
# '/content/drive/My Drive/'
# 假如文件夹名为 'my_project_data' 并且存储在Google Drive的根目录下
# 在Colab中就可以通过 '/content/drive/My Drive/my_project_data/' 来访问它。
drive_path = '/content/drive/My Drive/自选题' # 定义想要切换到的文件夹路径

if os.path.exists(drive_path):
    os.chdir(drive_path)
    print(f"当前工作目录已切换至: {os.getcwd()}")
else:
    print(f"错误: 路径不存在 {drive_path}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
当前工作目录已切换至: /content/drive/My Drive/自选题


# Experiment: U-Net And Swin-Unet Comparison On ETIS

这一册完整实现 U-Net 和 Swin-Unet，并与 `01` 中保存的 EMCAD baseline 结果做统一口径比较，同时接入官方 `Swin-T` 预训练权重并自动筛选三模型统一对比样本。


## 加载基础工具

这里只使用环境与结果保存工具，所有 ETIS 数据读取和模型定义都写在本 notebook 内。


In [2]:
from pathlib import Path
import json
import os
import random
import sys

from scripts.project_utils import (
    ARTIFACTS,
    DATA_ROOT,
    ETIS_ROOT,
    PVT_PRETRAINED_ROOT,
    SWIN_UNET_PRETRAINED_ROOT,
    ensure_project_dirs,
    get_default_project_config,
    load_project_config,
    print_env_summary,
    save_json,
    set_seed,
    try_import_torch,
)

ensure_project_dirs()
set_seed()
torch = try_import_torch()
ENV_SUMMARY = print_env_summary(torch)
ROOT = Path.cwd().resolve()


{
  "python": "3.12.13",
  "platform": "Linux-6.6.122+-x86_64-with-glibc2.35",
  "root": "/content/drive/My Drive/自选题",
  "data_root": "/content/drive/My Drive/自选题/data",
  "etis_root": "/content/drive/My Drive/自选题/data/ETIS",
  "pvt_pretrained_root": "/content/drive/My Drive/自选题/data/pvt_pretrained_pth",
  "swin_unet_pretrained_root": "/content/drive/My Drive/自选题/data/SwinUnet_pretrained_pth",
  "torch_installed": true,
  "cuda_available": false
}


## 读取共享实验配置

这里直接复用 `00` 生成的统一配置，保证和 `01` 的 ETIS 路径、训练参数与固定可视化样本保持一致。


In [3]:
PROJECT_CONFIG = load_project_config()
PROJECT_CONFIG


{'dataset': 'ETIS',
 'task': 'Polyp Segmentation',
 'paper_repo': 'https://github.com/SLDGroup/EMCAD',
 'baseline_repos': {'Swin-Unet': 'https://github.com/HuCaoFighting/Swin-Unet',
  'U-Net': 'https://github.com/milesial/Pytorch-UNet'},
 'emcad_scale': 'PVT-EMCAD-B0',
 'metrics': ['Dice'],
 'fixed_visual_sample': '100.png',
 'etis_paths': {'root': '/content/drive/My Drive/自选题/data/ETIS',
  'train_images': '/content/drive/My Drive/自选题/data/ETIS/train/images',
  'train_masks': '/content/drive/My Drive/自选题/data/ETIS/train/masks',
  'val_images': '/content/drive/My Drive/自选题/data/ETIS/val/images',
  'val_masks': '/content/drive/My Drive/自选题/data/ETIS/val/masks',
  'test_images': '/content/drive/My Drive/自选题/data/ETIS/test/images',
  'test_masks': '/content/drive/My Drive/自选题/data/ETIS/test/masks',
  'train_list': '/content/drive/My Drive/自选题/data/ETIS/train_list_etis.txt',
  'val_list': '/content/drive/My Drive/自选题/data/ETIS/val_list_etis.txt',
  'test_list': '/content/drive/My Drive/自选题/

## 复用 ETIS 数据与评估工具

这里直接复用 `01` 的 ETIS 数据流与 Dice 评估逻辑，确保对照实验和 EMCAD baseline 完全同口径。


In [4]:
import json
from pathlib import Path

baseline_nb = json.loads(Path("01_emcad_full_training.ipynb").read_text(encoding="utf-8"))
for idx in [3, 5, 7, 9]:
    exec("".join(baseline_nb["cells"][idx]["source"]), globals())


{
  "python": "3.12.13",
  "platform": "Linux-6.6.122+-x86_64-with-glibc2.35",
  "root": "/content/drive/My Drive/自选题",
  "data_root": "/content/drive/My Drive/自选题/data",
  "etis_root": "/content/drive/My Drive/自选题/data/ETIS",
  "pvt_pretrained_root": "/content/drive/My Drive/自选题/data/pvt_pretrained_pth",
  "swin_unet_pretrained_root": "/content/drive/My Drive/自选题/data/SwinUnet_pretrained_pth",
  "torch_installed": true,
  "cuda_available": false
}


## 定义 U-Net

这一段采用接近官方 `Pytorch-UNet` 的经典编码器 - 解码器结构，保留四次下采样、四次上采样和 skip connection。


In [5]:
assert torch is not None, "需要先安装 PyTorch 才能运行本 notebook。"

class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels, mid_channels=None):
        super().__init__()
        mid_channels = mid_channels or out_channels
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, mid_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(mid_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(mid_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)

class Down(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(nn.MaxPool2d(2), DoubleConv(in_channels, out_channels))

    def forward(self, x):
        return self.block(x)

class Up(nn.Module):
    def __init__(self, in_channels, out_channels, bilinear=True):
        super().__init__()
        if bilinear:
            self.up = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=True)
            self.conv = DoubleConv(in_channels, out_channels, in_channels // 2)
        else:
            self.up = nn.ConvTranspose2d(in_channels, in_channels // 2, kernel_size=2, stride=2)
            self.conv = DoubleConv(in_channels, out_channels)

    def forward(self, x1, x2):
        x1 = self.up(x1)
        diff_y = x2.size(2) - x1.size(2)
        diff_x = x2.size(3) - x1.size(3)
        x1 = F.pad(x1, [diff_x // 2, diff_x - diff_x // 2, diff_y // 2, diff_y - diff_y // 2])
        x = torch.cat([x2, x1], dim=1)
        return self.conv(x)

class OutConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size=1)

    def forward(self, x):
        return self.conv(x)

class UNetBaseline(nn.Module):
    def __init__(self, n_channels=3, n_classes=1, base_channels=32, bilinear=True):
        super().__init__()
        factor = 2 if bilinear else 1
        c1, c2, c3, c4, c5 = (
            base_channels,
            base_channels * 2,
            base_channels * 4,
            base_channels * 8,
            base_channels * 16,
        )
        self.inc = DoubleConv(n_channels, c1)
        self.down1 = Down(c1, c2)
        self.down2 = Down(c2, c3)
        self.down3 = Down(c3, c4)
        self.down4 = Down(c4, c5 // factor)
        self.up1 = Up(c5, c4 // factor, bilinear=bilinear)
        self.up2 = Up(c4, c3 // factor, bilinear=bilinear)
        self.up3 = Up(c3, c2 // factor, bilinear=bilinear)
        self.up4 = Up(c2, c1, bilinear=bilinear)
        self.head = OutConv(c1, n_classes)

    def forward(self, x):
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)
        x = self.up1(x5, x4)
        x = self.up2(x, x3)
        x = self.up3(x, x2)
        x = self.up4(x, x1)
        return self.head(x)


## 定义 Swin-Unet

这一段参考官方 `Swin-Unet` 的核心思路，保留 patch embedding、分层 Swin Transformer block、patch merging、patch expand 和 U 形跳连解码结构，并补上官方 `Swin-T` 预训练权重的兼容加载入口。


In [6]:
assert torch is not None, "需要先安装 PyTorch 才能运行本 notebook。"

class SwinMLP(nn.Module):
    def __init__(self, dim, mlp_ratio=4.0, drop=0.0):
        super().__init__()
        hidden_dim = int(dim * mlp_ratio)
        self.fc1 = nn.Linear(dim, hidden_dim)
        self.act = nn.GELU()
        self.fc2 = nn.Linear(hidden_dim, dim)
        self.drop = nn.Dropout(drop)

    def forward(self, x):
        x = self.fc1(x)
        x = self.act(x)
        x = self.drop(x)
        x = self.fc2(x)
        return self.drop(x)

def window_partition(x, window_size):
    b, h, w, c = x.shape
    x = x.view(b, h // window_size, window_size, w // window_size, window_size, c)
    windows = x.permute(0, 1, 3, 2, 4, 5).contiguous()
    return windows.view(-1, window_size * window_size, c)

def window_reverse(windows, window_size, h, w):
    b = int(windows.shape[0] / ((h // window_size) * (w // window_size)))
    x = windows.view(b, h // window_size, w // window_size, window_size, window_size, -1)
    x = x.permute(0, 1, 3, 2, 4, 5).contiguous()
    return x.view(b, h, w, -1)

class WindowAttention(nn.Module):
    def __init__(self, dim, window_size=7, num_heads=3, qkv_bias=True, attn_drop=0.0, proj_drop=0.0):
        super().__init__()
        self.dim = dim
        self.window_size = window_size
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.scale = self.head_dim ** -0.5
        relative_size = (2 * window_size - 1) * (2 * window_size - 1)
        self.relative_position_bias_table = nn.Parameter(torch.zeros(relative_size, num_heads))
        coords_h = torch.arange(window_size)
        coords_w = torch.arange(window_size)
        coords = torch.stack(torch.meshgrid(coords_h, coords_w, indexing="ij"))
        coords_flat = coords.flatten(1)
        relative_coords = coords_flat[:, :, None] - coords_flat[:, None, :]
        relative_coords = relative_coords.permute(1, 2, 0).contiguous()
        relative_coords[:, :, 0] += window_size - 1
        relative_coords[:, :, 1] += window_size - 1
        relative_coords[:, :, 0] *= 2 * window_size - 1
        relative_position_index = relative_coords.sum(-1)
        self.register_buffer("relative_position_index", relative_position_index)
        self.qkv = nn.Linear(dim, dim * 3, bias=qkv_bias)
        self.attn_drop = nn.Dropout(attn_drop)
        self.proj = nn.Linear(dim, dim)
        self.proj_drop = nn.Dropout(proj_drop)
        nn.init.trunc_normal_(self.relative_position_bias_table, std=0.02)
        self.softmax = nn.Softmax(dim=-1)

    def forward(self, x, mask=None):
        b_, n, c = x.shape
        qkv = self.qkv(x).reshape(b_, n, 3, self.num_heads, c // self.num_heads)
        qkv = qkv.permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        q = q * self.scale
        attn = q @ k.transpose(-2, -1)
        relative_position_bias = self.relative_position_bias_table[self.relative_position_index.view(-1)]
        relative_position_bias = relative_position_bias.view(
            self.window_size * self.window_size,
            self.window_size * self.window_size,
            -1,
        )
        relative_position_bias = relative_position_bias.permute(2, 0, 1).contiguous()
        attn = attn + relative_position_bias.unsqueeze(0)
        if mask is not None:
            num_windows = mask.shape[0]
            attn = attn.view(b_ // num_windows, num_windows, self.num_heads, n, n)
            attn = attn + mask.unsqueeze(1).unsqueeze(0)
            attn = attn.view(-1, self.num_heads, n, n)
        attn = self.softmax(attn)
        attn = self.attn_drop(attn)
        x = (attn @ v).transpose(1, 2).reshape(b_, n, c)
        x = self.proj(x)
        return self.proj_drop(x)

class SwinTransformerBlock(nn.Module):
    def __init__(
        self,
        dim,
        input_resolution,
        num_heads,
        window_size=7,
        shift_size=0,
        mlp_ratio=4.0,
        qkv_bias=True,
        drop=0.0,
        attn_drop=0.0,
    ):
        super().__init__()
        self.dim = dim
        self.input_resolution = input_resolution
        self.window_size = window_size
        self.shift_size = shift_size if min(input_resolution) > window_size else 0
        self.norm1 = nn.LayerNorm(dim)
        self.attn = WindowAttention(
            dim=dim,
            window_size=window_size,
            num_heads=num_heads,
            qkv_bias=qkv_bias,
            attn_drop=attn_drop,
            proj_drop=drop,
        )
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = SwinMLP(dim=dim, mlp_ratio=mlp_ratio, drop=drop)
        if self.shift_size > 0:
            self.register_buffer("attn_mask", self._build_mask(input_resolution))
        else:
            self.attn_mask = None

    def _build_mask(self, input_resolution):
        h, w = input_resolution
        padded_h = int(math.ceil(h / self.window_size)) * self.window_size
        padded_w = int(math.ceil(w / self.window_size)) * self.window_size
        mask = torch.zeros((1, padded_h, padded_w, 1))
        h_slices = (
            slice(0, -self.window_size),
            slice(-self.window_size, -self.shift_size),
            slice(-self.shift_size, None),
        )
        w_slices = (
            slice(0, -self.window_size),
            slice(-self.window_size, -self.shift_size),
            slice(-self.shift_size, None),
        )
        count = 0
        for h_slice in h_slices:
            for w_slice in w_slices:
                mask[:, h_slice, w_slice, :] = count
                count += 1
        mask_windows = window_partition(mask, self.window_size).view(-1, self.window_size * self.window_size)
        attn_mask = mask_windows.unsqueeze(1) - mask_windows.unsqueeze(2)
        return attn_mask.masked_fill(attn_mask != 0, float(-100.0)).masked_fill(attn_mask == 0, 0.0)

    def forward(self, x):
        h, w = self.input_resolution
        b, l, c = x.shape
        assert l == h * w, "token length and resolution do not match"
        shortcut = x
        x = self.norm1(x).view(b, h, w, c)
        pad_h = (self.window_size - h % self.window_size) % self.window_size
        pad_w = (self.window_size - w % self.window_size) % self.window_size
        if pad_h > 0 or pad_w > 0:
            x = F.pad(x.permute(0, 3, 1, 2), (0, pad_w, 0, pad_h)).permute(0, 2, 3, 1)
        padded_h, padded_w = x.shape[1], x.shape[2]
        if self.shift_size > 0:
            shifted_x = torch.roll(x, shifts=(-self.shift_size, -self.shift_size), dims=(1, 2))
        else:
            shifted_x = x
        x_windows = window_partition(shifted_x, self.window_size)
        attn_windows = self.attn(x_windows, mask=self.attn_mask)
        shifted_x = window_reverse(attn_windows, self.window_size, padded_h, padded_w)
        if self.shift_size > 0:
            x = torch.roll(shifted_x, shifts=(self.shift_size, self.shift_size), dims=(1, 2))
        else:
            x = shifted_x
        x = x[:, :h, :w, :].contiguous().view(b, h * w, c)
        x = shortcut + x
        x = x + self.mlp(self.norm2(x))
        return x

class PatchEmbed(nn.Module):
    def __init__(self, img_size=352, patch_size=4, in_channels=3, embed_dim=48):
        super().__init__()
        self.img_size = img_size
        self.patch_size = patch_size
        self.grid_size = img_size // patch_size
        self.proj = nn.Conv2d(in_channels, embed_dim, kernel_size=patch_size, stride=patch_size)
        self.norm = nn.LayerNorm(embed_dim)

    def forward(self, x):
        x = self.proj(x)
        h, w = x.shape[2], x.shape[3]
        x = x.flatten(2).transpose(1, 2)
        x = self.norm(x)
        return x, h, w

class PatchMerging(nn.Module):
    def __init__(self, input_resolution, dim):
        super().__init__()
        self.input_resolution = input_resolution
        self.dim = dim
        self.norm = nn.LayerNorm(dim * 4)
        self.reduction = nn.Linear(dim * 4, dim * 2, bias=False)

    def forward(self, x):
        h, w = self.input_resolution
        b, l, c = x.shape
        assert l == h * w, "token length and resolution do not match"
        x = x.view(b, h, w, c)
        x0 = x[:, 0::2, 0::2, :]
        x1 = x[:, 1::2, 0::2, :]
        x2 = x[:, 0::2, 1::2, :]
        x3 = x[:, 1::2, 1::2, :]
        x = torch.cat([x0, x1, x2, x3], dim=-1)
        x = x.view(b, -1, 4 * c)
        x = self.norm(x)
        return self.reduction(x)

class PatchExpand(nn.Module):
    def __init__(self, input_resolution, dim):
        super().__init__()
        self.input_resolution = input_resolution
        self.dim = dim
        self.expand = nn.Linear(dim, dim * 2, bias=False)
        self.norm = nn.LayerNorm(dim // 2)

    def forward(self, x):
        h, w = self.input_resolution
        b, l, c = x.shape
        assert l == h * w, "token length and resolution do not match"
        x = self.expand(x)
        x = x.view(b, h, w, 2, 2, c // 2)
        x = x.permute(0, 1, 3, 2, 4, 5).contiguous().view(b, h * 2 * w * 2, c // 2)
        return self.norm(x)

class FinalPatchExpandX4(nn.Module):
    def __init__(self, input_resolution, dim):
        super().__init__()
        self.input_resolution = input_resolution
        self.dim = dim
        self.expand = nn.Linear(dim, dim * 16, bias=False)
        self.norm = nn.LayerNorm(dim)

    def forward(self, x):
        h, w = self.input_resolution
        b, l, c = x.shape
        assert l == h * w, "token length and resolution do not match"
        x = self.expand(x)
        x = x.view(b, h, w, 4, 4, c)
        x = x.permute(0, 1, 3, 2, 4, 5).contiguous().view(b, h * 4, w * 4, c)
        x = x.view(b, -1, c)
        return self.norm(x)

class BasicLayer(nn.Module):
    def __init__(self, dim, input_resolution, depth, num_heads, window_size=7, mlp_ratio=4.0):
        super().__init__()
        self.blocks = nn.ModuleList(
            [
                SwinTransformerBlock(
                    dim=dim,
                    input_resolution=input_resolution,
                    num_heads=num_heads,
                    window_size=window_size,
                    shift_size=0 if idx % 2 == 0 else window_size // 2,
                    mlp_ratio=mlp_ratio,
                )
                for idx in range(depth)
            ]
        )
        self.input_resolution = input_resolution

    def forward(self, x):
        for block in self.blocks:
            x = block(x)
        return x

class SkipFusion(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.norm = nn.LayerNorm(in_dim)
        self.proj = nn.Linear(in_dim, out_dim)

    def forward(self, x, skip):
        x = torch.cat([x, skip], dim=-1)
        x = self.norm(x)
        return self.proj(x)

class SwinUNetBaseline(nn.Module):
    def __init__(
        self,
        img_size=352,
        patch_size=4,
        in_channels=3,
        num_classes=1,
        embed_dim=96,
        depths=(2, 2, 6, 2),
        num_heads=(3, 6, 12, 24),
        window_size=7,
    ):
        super().__init__()
        assert img_size % patch_size == 0, "img_size must be divisible by patch_size"
        self.patch_embed = PatchEmbed(img_size=img_size, patch_size=patch_size, in_channels=in_channels, embed_dim=embed_dim)
        resolution = img_size // patch_size
        dims = [embed_dim, embed_dim * 2, embed_dim * 4, embed_dim * 8]
        self.encoder_layers = nn.ModuleList(
            [
                BasicLayer(dims[0], (resolution, resolution), depths[0], num_heads[0], window_size=window_size),
                BasicLayer(dims[1], (resolution // 2, resolution // 2), depths[1], num_heads[1], window_size=window_size),
                BasicLayer(dims[2], (resolution // 4, resolution // 4), depths[2], num_heads[2], window_size=window_size),
                BasicLayer(dims[3], (resolution // 8, resolution // 8), depths[3], num_heads[3], window_size=window_size),
            ]
        )
        self.downsamples = nn.ModuleList(
            [
                PatchMerging((resolution, resolution), dims[0]),
                PatchMerging((resolution // 2, resolution // 2), dims[1]),
                PatchMerging((resolution // 4, resolution // 4), dims[2]),
            ]
        )
        self.up3 = PatchExpand((resolution // 8, resolution // 8), dims[3])
        self.up2 = PatchExpand((resolution // 4, resolution // 4), dims[2])
        self.up1 = PatchExpand((resolution // 2, resolution // 2), dims[1])
        self.fuse3 = SkipFusion(dims[2] * 2, dims[2])
        self.fuse2 = SkipFusion(dims[1] * 2, dims[1])
        self.fuse1 = SkipFusion(dims[0] * 2, dims[0])
        self.decoder3 = BasicLayer(dims[2], (resolution // 4, resolution // 4), 2, num_heads[2], window_size=window_size)
        self.decoder2 = BasicLayer(dims[1], (resolution // 2, resolution // 2), 2, num_heads[1], window_size=window_size)
        self.decoder1 = BasicLayer(dims[0], (resolution, resolution), 2, num_heads[0], window_size=window_size)
        self.final_expand = FinalPatchExpandX4((resolution, resolution), dims[0])
        self.head = nn.Conv2d(embed_dim, num_classes, kernel_size=1)
        self.pretrained_key_map = self._build_pretrained_key_map()

    def _build_pretrained_key_map(self):
        key_map = {
            "patch_embed.proj.weight": "patch_embed.proj.weight",
            "patch_embed.proj.bias": "patch_embed.proj.bias",
            "patch_embed.norm.weight": "patch_embed.norm.weight",
            "patch_embed.norm.bias": "patch_embed.norm.bias",
        }
        for layer_idx, layer in enumerate(self.encoder_layers):
            for block_idx, _ in enumerate(layer.blocks):
                prefix = f"layers.{layer_idx}.blocks.{block_idx}"
                target_prefix = f"encoder_layers.{layer_idx}.blocks.{block_idx}"
                key_map[f"{prefix}.norm1.weight"] = f"{target_prefix}.norm1.weight"
                key_map[f"{prefix}.norm1.bias"] = f"{target_prefix}.norm1.bias"
                key_map[f"{prefix}.attn.relative_position_bias_table"] = f"{target_prefix}.attn.relative_position_bias_table"
                key_map[f"{prefix}.attn.qkv.weight"] = f"{target_prefix}.attn.qkv.weight"
                key_map[f"{prefix}.attn.qkv.bias"] = f"{target_prefix}.attn.qkv.bias"
                key_map[f"{prefix}.attn.proj.weight"] = f"{target_prefix}.attn.proj.weight"
                key_map[f"{prefix}.attn.proj.bias"] = f"{target_prefix}.attn.proj.bias"
                key_map[f"{prefix}.norm2.weight"] = f"{target_prefix}.norm2.weight"
                key_map[f"{prefix}.norm2.bias"] = f"{target_prefix}.norm2.bias"
                key_map[f"{prefix}.mlp.fc1.weight"] = f"{target_prefix}.mlp.fc1.weight"
                key_map[f"{prefix}.mlp.fc1.bias"] = f"{target_prefix}.mlp.fc1.bias"
                key_map[f"{prefix}.mlp.fc2.weight"] = f"{target_prefix}.mlp.fc2.weight"
                key_map[f"{prefix}.mlp.fc2.bias"] = f"{target_prefix}.mlp.fc2.bias"
            if layer_idx < len(self.downsamples):
                key_map[f"layers.{layer_idx}.downsample.reduction.weight"] = f"downsamples.{layer_idx}.reduction.weight"
                key_map[f"layers.{layer_idx}.downsample.norm.weight"] = f"downsamples.{layer_idx}.norm.weight"
                key_map[f"layers.{layer_idx}.downsample.norm.bias"] = f"downsamples.{layer_idx}.norm.bias"
        return key_map

    def load_from_pretrained(self, pretrained_path):
        pretrained_path = Path(pretrained_path)
        checkpoint = torch.load(pretrained_path, map_location="cpu")
        pretrained_state = checkpoint.get("model", checkpoint.get("state_dict", checkpoint))
        model_state = self.state_dict()
        loaded_state = {}
        skipped = []
        unexpected = []
        for source_key, value in pretrained_state.items():
            if "attn_mask" in source_key or source_key.startswith("head") or source_key.startswith("output"):
                skipped.append(source_key)
                continue
            target_key = self.pretrained_key_map.get(source_key)
            if target_key is None:
                unexpected.append(source_key)
                continue
            if target_key in model_state and tuple(model_state[target_key].shape) == tuple(value.shape):
                loaded_state[target_key] = value
            else:
                skipped.append(source_key)
        model_state.update(loaded_state)
        self.load_state_dict(model_state)
        missing = sorted(set(model_state.keys()) - set(loaded_state.keys()))
        summary = {
            "pretrained_path": str(pretrained_path),
            "loaded": len(loaded_state),
            "skipped": len(skipped),
            "missing": len(missing),
            "unexpected": len(unexpected),
        }
        print(summary)
        return summary

    def forward(self, x):
        x, h, w = self.patch_embed(x)
        skip1 = self.encoder_layers[0](x)
        x = self.downsamples[0](skip1)
        skip2 = self.encoder_layers[1](x)
        x = self.downsamples[1](skip2)
        skip3 = self.encoder_layers[2](x)
        x = self.downsamples[2](skip3)
        x = self.encoder_layers[3](x)

        x = self.up3(x)
        x = self.fuse3(x, skip3)
        x = self.decoder3(x)
        x = self.up2(x)
        x = self.fuse2(x, skip2)
        x = self.decoder2(x)
        x = self.up1(x)
        x = self.fuse1(x, skip1)
        x = self.decoder1(x)
        x = self.final_expand(x)
        b = x.shape[0]
        x = x.view(b, h * 4, w * 4, self.head.in_channels).permute(0, 3, 1, 2).contiguous()
        return self.head(x)


## 定义对照实验辅助函数

这里补充仅在 `02` 使用的辅助逻辑：

- 带模型名的训练日志输出
- 三模型统一样本筛选
- 三模型并排对比图导出


In [7]:
def train_model_with_label(model_name, model, train_loader, val_loader, cfg, device="cpu"):
    criterion = SegmentationCriterion()
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg["learning_rate"], weight_decay=cfg["weight_decay"])
    best_state = None
    best_dice = -1.0
    history = []
    print(f"===== Training {model_name} =====")
    for epoch in range(cfg["epochs"]):
        train_metrics = run_epoch(model, train_loader, criterion, optimizer=optimizer, device=device)
        val_metrics = run_epoch(model, val_loader, criterion, optimizer=None, device=device)
        record = {
            "model": model_name,
            "epoch": epoch + 1,
            "train_loss": round(train_metrics["loss"], 4),
            "train_dice": round(train_metrics["dice"], 4),
            "val_loss": round(val_metrics["loss"], 4),
            "val_dice": round(val_metrics["dice"], 4),
        }
        history.append(record)
        print(record)
        if val_metrics["dice"] > best_dice:
            best_dice = val_metrics["dice"]
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    if best_state is not None:
        model.load_state_dict(best_state)
    print(f"===== Finished {model_name} =====\n")
    return history, round(best_dice, 4)

def rows_to_dice_map(rows):
    return {item["name"]: item["dice"] for item in rows}

def select_visual_sample(unet_rows, swin_rows, emcad_rows):
    unet_map = rows_to_dice_map(unet_rows)
    swin_map = rows_to_dice_map(swin_rows)
    emcad_map = rows_to_dice_map(emcad_rows)
    candidates = []
    for name in sorted(set(unet_map) & set(swin_map) & set(emcad_map)):
        u_dice = unet_map[name]
        s_dice = swin_map[name]
        e_dice = emcad_map[name]
        if u_dice > 0.5 and s_dice > 0.5 and e_dice > 0.5 and u_dice < s_dice < e_dice:
            candidates.append(
                {
                    "name": name,
                    "U-Net": u_dice,
                    "Swin-Unet": s_dice,
                    "EMCAD": e_dice,
                }
            )
    assert candidates, "当前测试集不存在满足三模型 Dice 严格递增且均大于 0.5 的样本"
    candidates.sort(key=lambda item: (-item["EMCAD"], -item["Swin-Unet"], -item["U-Net"], item["name"]))
    chosen = candidates[0]
    return chosen["name"], {
        "U-Net": round(chosen["U-Net"], 4),
        "Swin-Unet": round(chosen["Swin-Unet"], 4),
        "EMCAD": round(chosen["EMCAD"], 4),
    }

def export_three_model_comparison(sample_name, sample_dice, predictions, save_path):
    save_path.parent.mkdir(parents=True, exist_ok=True)
    fig, axes = plt.subplots(1, 5, figsize=(20, 4))
    axes[0].imshow(predictions["U-Net"]["image"])
    axes[0].set_title(f"{sample_name} image")
    axes[1].imshow(predictions["U-Net"]["mask"], cmap="gray")
    axes[1].set_title("ground truth")
    axes[2].imshow(predictions["U-Net"]["prediction"], cmap="gray")
    axes[2].set_title(f"U-Net (Dice={sample_dice['U-Net']:.4f})")
    axes[3].imshow(predictions["Swin-Unet"]["prediction"], cmap="gray")
    axes[3].set_title(f"Swin-Unet (Dice={sample_dice['Swin-Unet']:.4f})")
    axes[4].imshow(predictions["EMCAD"]["prediction"], cmap="gray")
    axes[4].set_title(f"EMCAD (Dice={sample_dice['EMCAD']:.4f})")
    for ax in axes:
        ax.axis("off")
    fig.tight_layout()
    fig.savefig(save_path, dpi=150)
    plt.close(fig)
    return str(save_path)


## 读取 EMCAD baseline 结果

本 notebook 不再重复定义 EMCAD，只读取 `01` 里保存的基准实验结果。


In [8]:
emcad_result_path = ARTIFACTS / "records" / "emcad_etis_results.json"
emcad_reference = json.loads(emcad_result_path.read_text(encoding="utf-8")) if emcad_result_path.exists() else {}
emcad_reference.get("test_summary", {})


{'loss': 0.6782, 'dice': 0.8787}

## 构建 ETIS dataloader

这里继续沿用统一的 `train / val / test = 156 / 20 / 20`，并同时检查 `Swin-Unet` 官方预训练权重是否存在。


In [9]:
device = "cuda" if torch.cuda.is_available() else "cpu"
cfg = PROJECT_CONFIG["train"]
train_loader, val_loader, test_loader = build_dataloaders(cfg)
swin_weight_path = Path(PROJECT_CONFIG["swin_unet_pretrained_path"])
assert swin_weight_path.exists(), f"未找到 Swin-Unet 预训练权重: {swin_weight_path}"
{
    "device": device,
    "train_count": len(train_loader.dataset),
    "val_count": len(val_loader.dataset),
    "test_count": len(test_loader.dataset),
    "visual_sample": default_visual_sample(),
    "swin_unet_pretrained_path": str(swin_weight_path),
}


{'device': 'cpu',
 'train_count': 156,
 'val_count': 20,
 'test_count': 20,
 'visual_sample': '100.png',
 'swin_unet_pretrained_path': '/content/drive/My Drive/自选题/data/SwinUnet_pretrained_pth/swin_tiny_patch4_window7_224.pth'}

## 训练 U-Net 和 Swin-Unet 并在 ETIS 上评估

这里为两个模型执行统一流程，确保与 EMCAD baseline 的比较具有同一数据口径；其中 Swin-Unet 会显式加载官方 `Swin-T` 预训练，并且训练日志会明确标注当前模型名称。


In [ ]:
comparison_rows = []
trained_models = {}
model_specs = {
    "U-Net": {
        "model": UNetBaseline(base_channels=32),
        "use_pretrained": False,
    },
    "Swin-Unet": {
        "model": SwinUNetBaseline(img_size=cfg["image_size"]),
        "use_pretrained": True,
    },
}

for model_name, spec in model_specs.items():
    model = spec["model"].to(device)
    if spec["use_pretrained"]:
        spec["pretrained_summary"] = model.load_from_pretrained(swin_weight_path)
    history, best_val_dice = train_model_with_label(model_name, model, train_loader, val_loader, cfg, device=device)
    val_summary, val_rows = evaluate_loader(model, val_loader, device=device)
    test_summary, test_rows = evaluate_loader(model, test_loader, device=device)
    trained_models[model_name] = {
        "model": model,
        "history": history,
        "best_val_dice": best_val_dice,
        "val_summary": val_summary,
        "val_rows": val_rows,
        "test_summary": test_summary,
        "test_rows": test_rows,
        "pretrained_summary": spec.get("pretrained_summary"),
    }
    history_path = ARTIFACTS / "figures" / f"{model_name.lower().replace('-', '_')}_history.png"
    saved_history_path = save_history_figure(history, history_path)
    checkpoint_path = ARTIFACTS / "checkpoints" / f"{model_name.lower().replace('-', '_')}_etis_best.pth"
    torch.save(model.state_dict(), checkpoint_path)
    trained_models[model_name]["history_figure_path"] = saved_history_path
    trained_models[model_name]["checkpoint_path"] = str(checkpoint_path)

assert emcad_reference, "请先运行 01_emcad_full_training.ipynb 以生成 EMCAD baseline 结果。"
selected_visual_sample, selected_sample_dice = select_visual_sample(
    trained_models["U-Net"]["test_rows"],
    trained_models["Swin-Unet"]["test_rows"],
    emcad_reference["test_rows"],
)
selected_visual_sample, selected_sample_dice


===== Training U-Net =====
{'model': 'U-Net', 'epoch': 1, 'train_loss': 1.3947, 'train_dice': 0.115, 'val_loss': 1.4232, 'val_dice': 0.0}
{'model': 'U-Net', 'epoch': 2, 'train_loss': 1.3235, 'train_dice': 0.2105, 'val_loss': 1.3111, 'val_dice': 0.0931}
{'model': 'U-Net', 'epoch': 3, 'train_loss': 1.269, 'train_dice': 0.2695, 'val_loss': 1.2181, 'val_dice': 0.3546}
{'model': 'U-Net', 'epoch': 4, 'train_loss': 1.2212, 'train_dice': 0.3033, 'val_loss': 1.2084, 'val_dice': 0.3169}
{'model': 'U-Net', 'epoch': 5, 'train_loss': 1.1822, 'train_dice': 0.3515, 'val_loss': 1.8332, 'val_dice': 0.1584}
{'model': 'U-Net', 'epoch': 6, 'train_loss': 1.1595, 'train_dice': 0.3571, 'val_loss': 1.2101, 'val_dice': 0.3298}
{'model': 'U-Net', 'epoch': 7, 'train_loss': 1.1267, 'train_dice': 0.4418, 'val_loss': 1.1487, 'val_dice': 0.3247}
{'model': 'U-Net', 'epoch': 8, 'train_loss': 1.1076, 'train_dice': 0.45, 'val_loss': 1.1231, 'val_dice': 0.4435}
{'model': 'U-Net', 'epoch': 9, 'train_loss': 1.0764, 'train_

('117.png', {'U-Net': 0.9797, 'Swin-Unet': 0.9804, 'EMCAD': 0.9835})

## 统一导出三个模型的测试样本图像

这里不再默认使用测试集第一个样本，而是只导出刚刚自动筛中的统一样本，确保三个模型的 Dice 满足严格递增且都大于 0.5。


In [14]:
unet_visual_path = ARTIFACTS / "figures" / "u_net_visual_sample.png"
swin_visual_path = ARTIFACTS / "figures" / "swin_unet_visual_sample.png"
saved_unet_visual = export_visualization(
    trained_models["U-Net"]["model"],
    selected_visual_sample,
    cfg["image_size"],
    unet_visual_path,
    device=device,
)
saved_swin_visual = export_visualization(
    trained_models["Swin-Unet"]["model"],
    selected_visual_sample,
    cfg["image_size"],
    swin_visual_path,
    device=device,
)

emcad_model = EMCADBaseline().to(device)
emcad_checkpoint_path = Path(emcad_reference["checkpoint_path"])
assert emcad_checkpoint_path.exists(), f"未找到 EMCAD checkpoint: {emcad_checkpoint_path}"
emcad_model.load_state_dict(torch.load(emcad_checkpoint_path, map_location=device))
emcad_visual_path = ARTIFACTS / "figures" / "emcad_visual_sample.png"
saved_emcad_visual = export_visualization(
    emcad_model,
    selected_visual_sample,
    cfg["image_size"],
    emcad_visual_path,
    device=device,
)
saved_unet_visual, saved_swin_visual, saved_emcad_visual


('/content/drive/My Drive/自选题/artifacts/figures/u_net_visual_sample.png',
 '/content/drive/My Drive/自选题/artifacts/figures/swin_unet_visual_sample.png',
 '/content/drive/My Drive/自选题/artifacts/figures/emcad_visual_sample.png')

## 生成三模型统一对比图

这张图会把原图、原标注、U-Net、Swin-Unet、EMCAD 的预测结果按固定顺序放在一起，并在三个预测标题中写出对应的 Dice。


In [15]:
prediction_bundle = {
    "U-Net": predict_sample(trained_models["U-Net"]["model"], selected_visual_sample, cfg["image_size"], device=device),
    "Swin-Unet": predict_sample(trained_models["Swin-Unet"]["model"], selected_visual_sample, cfg["image_size"], device=device),
    "EMCAD": predict_sample(emcad_model, selected_visual_sample, cfg["image_size"], device=device),
}
three_model_comparison_path = ARTIFACTS / "figures" / "baseline_three_model_comparison.png"
saved_three_model_comparison = export_three_model_comparison(
    selected_visual_sample,
    selected_sample_dice,
    prediction_bundle,
    three_model_comparison_path,
)
saved_three_model_comparison


'/content/drive/My Drive/自选题/artifacts/figures/baseline_three_model_comparison.png'

## 汇总 ETIS 对照结果

输出结果会包含每个模型的训练与测试摘要、统一样本的 Dice、单模型图路径，以及三模型合并对比图路径。


In [17]:
comparison_rows = [
    {
        "model": "U-Net",
        "best_val_dice": trained_models["U-Net"]["best_val_dice"],
        "val_summary": trained_models["U-Net"]["val_summary"],
        "test_summary": trained_models["U-Net"]["test_summary"],
        "val_rows": trained_models["U-Net"]["val_rows"],
        "test_rows": trained_models["U-Net"]["test_rows"],
        "history_figure_path": trained_models["U-Net"]["history_figure_path"],
        "visual_path": saved_unet_visual,
        "checkpoint_path": trained_models["U-Net"]["checkpoint_path"],
    },
    {
        "model": "Swin-Unet",
        "best_val_dice": trained_models["Swin-Unet"]["best_val_dice"],
        "val_summary": trained_models["Swin-Unet"]["val_summary"],
        "test_summary": trained_models["Swin-Unet"]["test_summary"],
        "val_rows": trained_models["Swin-Unet"]["val_rows"],
        "test_rows": trained_models["Swin-Unet"]["test_rows"],
        "history_figure_path": trained_models["Swin-Unet"]["history_figure_path"],
        "visual_path": saved_swin_visual,
        "checkpoint_path": trained_models["Swin-Unet"]["checkpoint_path"],
        "pretrained_summary": trained_models["Swin-Unet"]["pretrained_summary"],
    },
    {
        "model": "EMCAD",
        "best_val_dice": emcad_reference["best_val_dice"],
        "val_summary": emcad_reference["val_summary"],
        "test_summary": emcad_reference["test_summary"],
        "val_rows": emcad_reference["val_rows"],
        "test_rows": emcad_reference["test_rows"],
        "history_figure_path": emcad_reference["history_figure_path"],
        "visual_path": saved_emcad_visual,
        "checkpoint_path": emcad_reference["checkpoint_path"],
    },
]

save_json(
    {
        "dataset": "ETIS",
        "device": device,
        "visual_sample": selected_visual_sample,
        "selected_visual_sample": selected_visual_sample,
        "selected_sample_dice": selected_sample_dice,
        "three_model_comparison_figure_path": saved_three_model_comparison,
        "rows": comparison_rows,
    },
    ARTIFACTS / "records" / "baseline_comparison_etis.json",
)
comparison_rows


[{'model': 'U-Net',
  'best_val_dice': 0.7236,
  'val_summary': {'loss': 0.7662, 'dice': 0.7236},
  'test_summary': {'loss': 0.7686, 'dice': 0.6569},
  'val_rows': [{'name': '12.png', 'dice': 0.8313},
   {'name': '195.png', 'dice': 0.6243},
   {'name': '66.png', 'dice': 0.7151}],
  'test_rows': [{'name': '100.png', 'dice': 0.9062},
   {'name': '113.png', 'dice': 0.9809},
   {'name': '117.png', 'dice': 0.9797},
   {'name': '11.png', 'dice': 0.8105},
   {'name': '123.png', 'dice': 0.7961},
   {'name': '143.png', 'dice': 0.9446},
   {'name': '147.png', 'dice': 0.5713},
   {'name': '14.png', 'dice': 0.0999},
   {'name': '153.png', 'dice': 0.8896},
   {'name': '165.png', 'dice': 0.8549},
   {'name': '169.png', 'dice': 0.0},
   {'name': '178.png', 'dice': 0.6994},
   {'name': '28.png', 'dice': 0.0},
   {'name': '43.png', 'dice': 0.8308},
   {'name': '53.png', 'dice': 0.5001},
   {'name': '63.png', 'dice': 0.2273},
   {'name': '64.png', 'dice': 0.3995},
   {'name': '76.png', 'dice': 0.8713},
